In [1]:
import os
import sqlite3
import pandas as pd
import geopandas as gpd
import random
import time

In [2]:
'''DEFINE THE FILEPATHS TO BE USED IN THE SCRIPT'''
conndir = 'C:/IMAGE/Foliage/image_api/data/subgrids.db'
targetdir = "C:/IMAGE/Foliage/image_api/subgrids/"
filename = "SUBGRID_DATA_IMAGEBASED_MUM"

In [3]:
#list the files
filelist = os.listdir(targetdir)
print("Files involved: ") 
for file in filelist:
    print(file)

#read them into pandas
#df_list = [pd.read_table(targetdir+file) for file in filelist]

Files involved: 
SUBGRID_DATA_IMAGEBASED_BLR.txt
SUBGRID_DATA_IMAGEBASED_DEL.txt
SUBGRID_DATA_IMAGEBASED_MUM.txt


In [4]:
import json
import requests

def get_resource(r):
    resourceSets = json.loads(r.text)
    try:
        for resource in resourceSets['resourceSets']:
            return [rsc for rsc in resource['resources']]
    except KeyError:
        try:
            resourcesSets = resourceSets['Response']['ResourceSets']
            resourceSet = resourcesSets['ResourceSet']
            if isinstance(resourceSet, dict):
                return resourceSet['Resources']
            elif isinstance(resourceSet, list):
                return [rsc for rsc in resourceSet['Resources']]
        except KeyError:
            print(KeyError)

def get_elevation(lat, lon):
    key = 'Ap2jFsGK3XZxoJYoaoE8pMLEk2B4x8OqgkolcqfDPwN-24_V9Vd__m5YOCyXPXch'
    point_query = f"{lat},{lon}"
    api_query = f'http://dev.virtualearth.net/REST/v1/Elevation/List?points={point_query}&heights=sealevel&key={key}'
    
    r = requests.get(api_query)
    if r.status_code != 200:
        raise r.raise_for_status()

    resource = get_resource(r)
    elevation = resource[0]['elevations'][0]

    return elevation

In [5]:
df = pd.read_table(targetdir+filename+".txt")
if "Unnamed: 0" in df.columns:
    df.drop('Unnamed: 0', axis=1, inplace=True)
df.head()

,IMAGE_NAME,PT_LAT,PT_LON,BUILD_PROB,FOLIAGE_PROB,CLASS
0,Mumbai.0.tif,19.317453,72.773943,0.000178,0.059522,Other
1,Mumbai.0.tif,19.317453,72.774392,0.000179,0.054801,Other
2,Mumbai.0.tif,19.317453,72.774841,0.000171,0.055578,Other
3,Mumbai.0.tif,19.317453,72.775290,0.000203,0.052009,Other
4,Mumbai.0.tif,19.317453,72.775739,0.000186,0.059955,Other


In [6]:
'''
list_amsl = []
count = 0
batch_size = 10
for idx in df.index:
    latitude = df["PT_LAT"][idx]
    longitude = df["PT_LON"][idx]

    elevation = get_elevation(latitude, longitude)
    list_amsl.append(elevation)

    if (idx+1) % batch_size == 0:
        time.sleep(random.randint(20, 60))

    print("HI")


df["AMSL"] = pd.Series(list_amsl).values
'''

'\nlist_amsl = []\ncount = 0\nbatch_size = 10\nfor idx in df.index:\n    latitude = df["PT_LAT"][idx]\n    longitude = df["PT_LON"][idx]\n\n    elevation = get_elevation(latitude, longitude)\n    list_amsl.append(elevation)\n\n    if (idx+1) % batch_size == 0:\n        time.sleep(random.randint(20, 60))\n\n    print("HI")\n\n\ndf["AMSL"] = pd.Series(list_amsl).values\n'

In [7]:
conn = sqlite3.connect(conndir)
try:
    df.to_sql(name=filename,con=conn,index=False,if_exists='append')
    conn.commit()
    conn.close()
except sqlite3.OperationalError as e:
    print("Error connecting to database:", e)